In [1]:
# Bibliothèques

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ex24_prix_agricoles.csv')
df.head()

Mounted at /content/drive


,record_id,governorate,city,region,year,month,date,latitude,longitude,utm32_easting,...,observed_lag_1,observed_lag_2,observed_lag_3,observed_lag_6,observed_lag_12,rolling_mean_3,rolling_mean_6,target_stage1_tabular_score,target_stage2_lstm_score,lstm_agri_price_next_3m
0,EX24_Ariana_2017_01,Ariana,Ariana,Nord-Est,2017,1,2017-01-15,36.801522,10.192484,606379.84,...,41.29,50.48,64.35,83.22,24.04,52.040000,67.230000,17.16,47.13,47.13
1,EX24_Ariana_2017_02,Ariana,Ariana,Nord-Est,2017,2,2017-02-15,36.832852,10.181435,605351.20,...,37.05,41.29,50.48,84.56,31.94,42.940000,59.535000,46.36,60.05,60.05
2,EX24_Ariana_2017_03,Ariana,Ariana,Nord-Est,2017,3,2017-03-15,36.906929,10.127179,600415.86,...,29.34,37.05,41.29,79.48,50.99,35.893333,50.331667,51.52,73.87,73.87
3,EX24_Ariana_2017_04,Ariana,Ariana,Nord-Est,2017,4,2017-04-15,36.865477,10.184440,605574.33,...,44.78,29.34,37.05,64.35,48.46,37.056667,44.548333,66.19,70.74,70.74
4,EX24_Ariana_2017_05,Ariana,Ariana,Nord-Est,2017,5,2017-05-15,36.901159,10.121643,599930.23,...,62.46,44.78,29.34,50.48,58.48,45.526667,44.233333,81.71,75.44,75.44


In [ ]:
drop_cols = ['record_id','governorate','city','region','date','data_nature','target_stage1_tabular_score','target_stage2_lstm_score','lstm_agri_price_next_3m']

feature_cols = [c for c in df.columns if c not in drop_cols and df[c].dtype != object]
target_col = 'target_stage1_tabular_score'

X = df[feature_cols]
y = df[target_col]

# split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# entrenement
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# metriques
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

# importance des features
fi = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_})
print(fi.sort_values('importance', ascending=False).head(10).to_string(index=False))



MAE  : 9.2318
RMSE : 11.5407
R²   : 0.6542
               feature  importance
         temperature_c    0.461435
drought_pressure_score    0.117288
        observed_index    0.052379
                 month    0.048725
          humidity_pct    0.036821
        observed_lag_3    0.034536
            ndvi_score    0.025580
        rolling_mean_6    0.021642
        observed_lag_2    0.020956
        observed_lag_6    0.020692


# Partie B

In [ ]:
# choix du nbre de clusters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

sil_scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    sil_scores[k] = silhouette_score(X_scaled, km.fit_predict(X_scaled))
best_k = max(sil_scores, key=sil_scores.get)
print(f"Meilleur k : {best_k} ")
    # ajout de la var cluster
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_scaled)

# modele avec cluster
feature_cols_clust = feature_cols + ['cluster']
X_clust = df[feature_cols_clust]
y       = df['target_stage1_tabular_score']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_clust, y, test_size=0.2, random_state=42, shuffle=False
)
rf_clust = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_clust.fit(X_tr, y_tr)
y_pred = rf_clust.predict(X_te)

print(f"MAE  : {mean_absolute_error(y_te, y_pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_te, y_pred)):.4f}")
print(f"R²   : {r2_score(y_te, y_pred):.4f}")

Meilleur k : 6 
MAE  : 9.2917
RMSE : 11.5686
R²   : 0.6833


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

# ── Préparation des données ──────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Fonction de construction du MLP ─────────────────────────────
def build_mlp(n_layers=2, n_units=64, dropout=0.2, lr=0.001):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_s.shape[1],)))
    for _ in range(n_layers):
        model.add(layers.Dense(n_units, activation='relu'))
        model.add(layers.BatchNormalization())   # stabilise l'entraînement
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))                  # sortie régression
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss='mse',
        metrics=['mae']
    )
    return model

# ═══════════════════════════════════════════════════════════════
# MLP NON TUNÉ — architecture simple par défaut
# ═══════════════════════════════════════════════════════════════
print("=== MLP NON TUNÉ (2 couches, 64 neurones, dropout=0.2, lr=0.001, bs=32) ===")

mlp_base = build_mlp(n_layers=2, n_units=64, dropout=0.2, lr=0.001)
mlp_base.fit(
    X_train_s, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

y_pred_base = mlp_base.predict(X_test_s).flatten()
mae_b  = mean_absolute_error(y_test, y_pred_base)
rmse_b = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_b   = r2_score(y_test, y_pred_base)

print(f"MAE  : {mae_b:.4f}")
print(f"RMSE : {rmse_b:.4f}")
print(f"R²   : {r2_b:.4f}")

# ═══════════════════════════════════════════════════════════════
# MLP TUNÉ — recherche sur 3 hyperparamètres + early stopping
# Hyperparamètres testés :
#   1. n_layers   : profondeur du réseau       (2 ou 3)
#   2. n_units    : largeur de chaque couche   (64 ou 128)
#   3. dropout    : régularisation             (0.1 ou 0.2)
#   4. lr         : vitesse d'apprentissage    (0.001 ou 0.0005)
#   5. batch_size : taille des mini-lots       (32 ou 64)
# ═══════════════════════════════════════════════════════════════
print("\n=== TUNING MANUEL (7 configurations) ===")

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

configs = [
    {'n_layers':2, 'n_units':64,  'dropout':0.1, 'lr':0.001,  'batch_size':32},
    {'n_layers':2, 'n_units':128, 'dropout':0.1, 'lr':0.001,  'batch_size':32},
    {'n_layers':3, 'n_units':64,  'dropout':0.2, 'lr':0.001,  'batch_size':64},
    {'n_layers':3, 'n_units':128, 'dropout':0.1, 'lr':0.001,  'batch_size':32},
    {'n_layers':3, 'n_units':128, 'dropout':0.2, 'lr':0.0005, 'batch_size':32},
    {'n_layers':2, 'n_units':128, 'dropout':0.3, 'lr':0.0005, 'batch_size':64},
    {'n_layers':3, 'n_units':64,  'dropout':0.1, 'lr':0.0005, 'batch_size':32},
]

best_r2, best_params, best_pred = -np.inf, None, None
results = []

for cfg in configs:
    m = build_mlp(cfg['n_layers'], cfg['n_units'], cfg['dropout'], cfg['lr'])
    m.fit(
        X_train_s, y_train,
        epochs=150,
        batch_size=cfg['batch_size'],
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )
    pred = m.predict(X_test_s).flatten()
    r2   = r2_score(y_test, pred)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results.append({**cfg, 'R²': round(r2,4), 'MAE': round(mae,4), 'RMSE': round(rmse,4)})
    print(f"  layers={cfg['n_layers']} units={cfg['n_units']} drop={cfg['dropout']} "
          f"lr={cfg['lr']} bs={cfg['batch_size']} → R²={r2:.4f}")
    if r2 > best_r2:
        best_r2, best_params, best_pred = r2, cfg, pred

# ── Résultats finaux ─────────────────────────────────────────────
print("\n=== TABLEAU COMPLET DES CONFIGS ===")
res_df = pd.DataFrame(results).sort_values('R²', ascending=False)
print(res_df.to_string(index=False))

print(f"\nMeilleurs hyperparamètres : {best_params}")
print(f"   MAE  : {mean_absolute_error(y_test, best_pred):.4f}")
print(f"   RMSE : {np.sqrt(mean_squared_error(y_test, best_pred)):.4f}")
print(f"   R²   : {best_r2:.4f}")

# ── Bilan global ─────────────────────────────────────────────────
print("\nBILAN PARTIES A → C ")
print(f"{'Modèle':<35} {'R²':>8}")
print(f"{'Random Forest (sans cluster)':<35} {'0.6535':>8}")
print(f"{'Random Forest (avec cluster)':<35} {'0.6824':>8}")
print(f"{'MLP non tuné':<35} {r2_b:>8.4f}")
print(f"{'MLP tuné':<35} {best_r2:>8.4f}")

=== MLP NON TUNÉ (2 couches, 64 neurones, dropout=0.2, lr=0.001, bs=32) ===
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
MAE  : 9.0802
RMSE : 11.3165
R²   : 0.6675

=== TUNING MANUEL (7 configurations) ===
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
  layers=2 units=64 drop=0.1 lr=0.001 bs=32 → R²=0.6705
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
  layers=2 units=128 drop=0.1 lr=0.001 bs=32 → R²=-11.0168
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
  layers=3 units=64 drop=0.2 lr=0.001 bs=64 → R²=-11.5007
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
  layers=3 units=128 drop=0.1 lr=0.001 bs=32 → R²=-10.8999
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
  layers=3 units=128 drop=0.2 lr=0.0005 bs=32 → R²=-11.4757
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
  layers=2 units=128 drop=0.3 lr=0.0005 bs=64 → R²=-11.6302
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
  layers=3 units=64 drop=0.1 lr=0.0005 bs=32 → R²=-11.5493

=== TABLEAU COMPLET DES CONFIGS ===
 n_layers  n_units  dropout     lr  batch_size       R²     MAE    RMSE
     